## Model Drift

A deployed model that performed well starts performing worse over time — without anyone changing the model.
The model didn't change. The world did.
Silent failure — no errors, no crashes, just wrong answers delivered confidently.

## How it actually works

**Two types of drift:**

**Data drift:**
The incoming data starts looking different from the data the model was trained on.
The model encounters inputs it never saw during training.
Predictions become unreliable.

Example: fraud detection model trained on 2023 invoice patterns.
In 2025, vendors switch to new payment methods and invoice formats.
The model has never seen these patterns. Accuracy drops silently.

**Concept drift:**
The relationship between inputs and correct outputs changes over time.
The data looks the same but the correct answer has changed.

Example: approval routing model learned invoices over $50,000 go to senior management.
Company restructures — now $50,000 goes to middle management.
The model keeps routing to senior management.
Data unchanged. Business rule changed. Model is wrong.

**Why it is dangerous:**
Unlike a crashed application, drifted models produce output.
Confident, formatted, wrong output.
You only discover it when someone notices accuracy dropped —
often weeks or months after go-live.

## Where it breaks or gets dangerous

**Silent period-end failures:**
A D365 AI assistant trained on pre-restructure data continues giving wrong routing advice.
Finance team follows the advice. Wrong approvals accumulate.
Audit finds the issue months later.

**Detecting drift:**
- Monitor prediction distributions over time — are output patterns changing?
- Track accuracy against ground truth when feedback is available
- Compare incoming data statistics against training data statistics
- Set alerts when key metrics drop below threshold

**Fixing drift:**
- Retrain model on fresh data including recent examples
- Fine-tune on recent examples without full retraining
- Build a feedback loop — corrections feed back into retraining pipeline
- Continuous Training (CT) — automate retraining when drift is detected

**The hardest part:**
You need ground truth to measure drift.
For some problems — fraud detection, approval routing — ground truth arrives days or weeks later.
The invoice flagged as fraud today may not be confirmed as fraud for two weeks.
You cannot measure accuracy in real time.
Design your monitoring to account for this lag.

In [ ]:
# Model Drift — detection illustration

import random

def simulate_model_predictions(period, drift=False):
    """
    Simulates model predictions over time.
    With drift, accuracy degrades gradually.
    """
    base_accuracy = 0.94
    
    if drift:
        # Accuracy degrades over time
        degradation = period * 0.03
        accuracy = max(base_accuracy - degradation, 0.60)
    else:
        # Small natural variance, no drift
        accuracy = base_accuracy + random.uniform(-0.02, 0.02)
    
    correct = int(accuracy * 100)
    incorrect = 100 - correct
    return accuracy, correct, incorrect

def detect_drift(accuracies, threshold=0.85):
    """Simple drift detection — alert when accuracy drops below threshold."""
    alerts = []
    for period, acc in enumerate(accuracies):
        if acc < threshold:
            alerts.append(f"Period {period + 1}: DRIFT ALERT — accuracy {acc:.0%} below threshold {threshold:.0%}")
    return alerts

print("=== Scenario 1: No Drift ===")
print(f"{'Period':<10} {'Accuracy':>10} {'Correct/100':>12} {'Status':>10}")
print("-" * 45)
stable_accuracies = []
for period in range(8):
    acc, correct, incorrect = simulate_model_predictions(period, drift=False)
    stable_accuracies.append(acc)
    print(f"{period+1:<10} {acc:>10.1%} {correct:>8}/100    {'✅ OK':>10}")

print()
print("=== Scenario 2: Data Drift Detected ===")
print(f"{'Period':<10} {'Accuracy':>10} {'Correct/100':>12} {'Status':>10}")
print("-" * 45)
drift_accuracies = []
for period in range(8):
    acc, correct, incorrect = simulate_model_predictions(period, drift=True)
    drift_accuracies.append(acc)
    status = "✅ OK" if acc >= 0.85 else "🚨 ALERT"
    print(f"{period+1:<10} {acc:>10.1%} {correct:>8}/100    {status:>10}")

print()
alerts = detect_drift(drift_accuracies)
if alerts:
    print("Drift Detection Alerts:")
    for alert in alerts:
        print(f"  {alert}")
    print()
    print("Action required: investigate data distribution changes and consider retraining.")

## My D365 analogy

Period-end reconciliation in D365 exists for exactly this reason.
You don't assume the system is correct forever.
You verify regularly — expected vs actual, threshold vs deviation, explain or correct.

Model monitoring is period-end reconciliation for AI systems.
You set the expected performance baseline at deployment.
You measure actual performance on a schedule.
When the gap exceeds tolerance — you investigate and correct.

Concept drift maps directly to a business rule change
that nobody updated in the system configuration.
The rules changed. The system kept running the old rules.
Every D365 consultant has seen this after a business restructure.
The fix is the same: identify the delta, update the configuration, retest.

In ML the configuration is the model weights.
Retraining is the equivalent of updating the configuration.

## LinkedIn post idea

In D365, we run period-end reconciliation.
Not because we expect the system to be wrong —
because we know the business changes and the system needs to keep up.

AI models have the same problem. It is called model drift.

A fraud detection model trained on last year's invoice patterns
will quietly become less accurate as vendor behaviour changes.
No error message. No crash. Just wrong answers with full confidence.

The fix is the same discipline we apply at period end:
measure expected vs actual, set thresholds, alert on deviation, correct and retest.

We already do this for financial postings.
We need to do it for every AI model we deploy.

Monitoring is not optional in production AI.
It is the period-end close.